IMPORT REQUIRED LIBRARIES

In [50]:
import optuna 
import optuna_dashboard
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split,KFold,cross_val_score,GridSearchCV
from xgboost import XGBClassifier
from sklearn.metrics import precision_score,recall_score,accuracy_score,confusion_matrix,classification_report
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt
import seaborn as sns

split data into training and testing

In [51]:
data=pd.read_csv(r"C:\Users\Avijit\Desktop\CANCER_PREDICTION\data\processed\cancer.csv")
x=data.iloc[:,:-1]
y=data['Diagnosis']
xtrain,xtest,ytrain,ytest=train_test_split(x,y,test_size=0.2,random_state=42)

model training

In [52]:
# xgb=XGBClassifier()
# xgb.fit(xtrain,ytrain)
# xgb.score(xtest,ytest)*100
# ypred=xgb.predict(xtest)
# xgb.score(xtrain,ytrain)*100
rfc=RandomForestClassifier(max_depth=10)
rfc.fit(xtrain,ytrain)
ypred=rfc.predict(xtest)
print("TRAINING SCORE:",rfc.score(xtrain,ytrain)*100,"TESTING SCORE:",rfc.score(xtest,ytest)*100)


TRAINING SCORE: 98.16666666666667 TESTING SCORE: 93.0


MODEL VISUALIZATION

In [ ]:
plt.figure(figsize=(40,35))
plot_tree(rfc.estimators_[0],feature_names=xtrain.columns,filled=True)
plt.show()

model tuning

In [ ]:
bestsplit,bestscore=0,0
for i in range(2,10):
    p=cross_val_score(RandomForestClassifier(),xtrain,ytrain,cv=KFold(n_splits=i)).mean()
    print
    if(p>bestscore):
        bestscore=p
        bestsplit=i
        

print("BESTSPLIT: ",bestsplit)
print("BESTSCORE: ",bestscore)

In [25]:
dict={
    "max_depth":(1,10)}
grd=GridSearchCV(RandomForestClassifier(),param_grid=dict,cv=5)
grd.fit(x,y)
print(grd.best_params_,grd.best_score_)

{'max_depth': 10} 0.9226666666666666


In [ ]:
def obj(trial):
    n_estimators=trial.suggest_int("n_estimators",1,20)
    criterion= trial.suggest_categorical('criterion',['gini', 'entropy', 'log_loss'])
    max_depth=trial.suggest_int('max_depth',1,20)
    rfc=RandomForestClassifier(
        n_estimators=n_estimators,
        criterion=criterion,
        max_depth=max_depth,
    )
    rfc.fit(xtrain,ytrain)
    Score=cross_val_score(RandomForestClassifier(),x,y,cv=KFold(n_splits=8)).mean()
    return Score
    
study=optuna.create_study(direction="maximize",study_name='cancer5',storage="sqlite:///dtr_study.db")
study.optimize(obj,n_trials=50)
study.best_params

model tuning dashboard

In [ ]:
optuna_dashboard.run_server("sqlite:///dtr_study.db")


model evaluation

In [55]:
print("PRECISION SCORE:",precision_score(ytest,ypred)*100)
print("RECALL SCORE:",recall_score(ytest,ypred)*100)
print("ACCURACY SCORE:",accuracy_score(ytest,ypred)*100)
print("CONFUSION MATRIX:\n",confusion_matrix(ytest,ypred))
print("CALSSIFICATION REPORT:\n",classification_report(ytest,ypred))

PRECISION SCORE: 94.39252336448598
RECALL SCORE: 87.06896551724138
ACCURACY SCORE: 93.0
CONFUSION MATRIX:
 [[178   6]
 [ 15 101]]
CALSSIFICATION REPORT:
               precision    recall  f1-score   support

           0       0.92      0.97      0.94       184
           1       0.94      0.87      0.91       116

    accuracy                           0.93       300
   macro avg       0.93      0.92      0.93       300
weighted avg       0.93      0.93      0.93       300



In [56]:
import joblib
joblib.dump(rfc,r"C:\Users\Avijit\Desktop\CANCER_PREDICTION\models\model.pkl")


['C:\\Users\\Avijit\\Desktop\\CANCER_PREDICTION\\models\\model.pkl']